# Building Agents with OpenAI · Structured paper extraction

Lesson 03 companion. Reuses the original PDF fixtures and CSV without overwriting them. [Structured outputs](https://developers.openai.com/api/docs/guides/structured-outputs).

In [ ]:
from pathlib import Path
import sys

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from course.config import Settings, api_client
settings = Settings()
LIVE = False  # Set True only when ready to send course fixtures to the API (billed).
print("Model:", settings.model, "| Live calls:", LIVE)
# The shared config loads .env; never print credentials.

## 1. Inspect the source
The page limit is explicit: up to 12 pages, each limited to 12,000 characters. Extraction requires a text PDF; scanned files need a separate OCR pipeline.

In [ ]:
from course.tools import paper_pages
pages = paper_pages(ROOT / "assets/paper.pdf")
print(pages["paper:1"][:1500])

## 2. Define the contract
Authors can be missing; an empty list is honest. Keep missing fields visible instead of inventing values.

In [ ]:
from course.schemas import PaperData
print(PaperData.model_json_schema())

## 3. Request, parse, and validate
The live helper uses `client.responses.parse(..., text_format=PaperData)`, rejects refusals/incomplete output, and checks the quote against the supplied page. It does not prove that the summary follows from that quote.

In [ ]:
import inspect
from course.extraction import extract_paper
print(inspect.getsource(extract_paper))
extracted = await extract_paper(live=LIVE, settings=settings)
print(extracted.model_dump_json(indent=2))

## 4. Save a reviewable extraction
Write generated data under ignored `outputs/`; the original `assets/papers_database.csv` stays a stable evaluation fixture.

In [ ]:
import csv
output = ROOT / "outputs/extraction.json"
output.parent.mkdir(exist_ok=True)
output.write_text(extracted.model_dump_json(indent=2) + "\n")
with (ROOT / "assets/papers_database.csv").open(newline="") as stream:
    records = list(csv.DictReader(stream))
print("Fixture records:", len(records), "| Saved:", output.name)

## Exercise
- Extract from `paper2.pdf` by adapting the helper; keep the original fixtures unchanged.
- Remove the author names from the supplied text and require explicit missing fields.
- Construct a schema-valid answer with an invented claim. Explain why validation alone accepts it.
- Compare the returned quote with the PDF, then review whether the summary follows from it.

Checkpoint: a schema enforces shape; sources establish evidence; evaluation checks behavior.